# Máscaras de referência por fonte de ground truth (estágio 02)

Adquire, para cada fonte de ground truth habilitada no `src/config.yaml` (MapBiomas, AlphaEarth), uma máscara binária de café em resolução de 10 m via Google Earth Engine, exportando cada fonte em GeoTIFF para o armazenamento canônico em `MyDrive/tcc/data/interim/<fonte>/`. Execuções repetidas reutilizam as máscaras já exportadas (idempotência) e o mosaico do estágio 01 é apenas verificado.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt`, garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)


## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`.

In [ ]:
# Importa o pacote compartilhado, identifica a plataforma e resolve os caminhos de armazenamento.
from src import io
from src.config import get_config

platform = io.detect_platform()
storage_paths = io.resolve_storage_paths()
config = get_config()
print(f"Plataforma: {platform}")
print(f"Máscaras por fonte: {storage_paths['data_interim']}/<fonte>")
print(f"Figuras: {storage_paths['artifacts_figures']}")


## Autenticação do Earth Engine

Autentica e inicializa o GEE com a conta principal: fluxo interativo no Colab; no Kaggle, carrega a credencial da variável `GEE_CREDENTIALS`. A lógica de plataforma fica isolada em `src/`.

In [ ]:
# Autentica e inicializa o Earth Engine com a conta principal.
from src.utils import authenticate_gee

authenticate_gee()
print("Earth Engine autenticado e inicializado.")


## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, garantindo o mesmo protocolo de execução nas duas plataformas.

In [ ]:
# Fixa sementes e flags determinísticas do PyTorch de acordo com a configuração.
from src.utils import set_all_seeds, set_deterministic_flags

set_all_seeds(config["reproducibility"]["seed"])
set_deterministic_flags()
print(f"Seed fixada: {config['reproducibility']['seed']}")


## Área de estudo (AOI) a partir da malha IBGE

Lê a malha vetorial das Regiões Geográficas Imediatas de Minas Gerais (dado de referência versionado em `data/external/`), filtra a região `310044` (Guaxupé), reprojeta para EPSG:4326 e converte o polígono em geometria do Earth Engine.

In [ ]:
# Carrega a malha IBGE, filtra a RGI 310044 e converte o AOI para geometria do GEE.
from src.data import gee_client

mesh_path = workspace / config["aoi"]["mesh_path"]
aoi_geometry, aoi_gdf = gee_client.load_aoi_geometry(
    mesh_path, config["aoi"]["region_code"]
)
aoi_row = aoi_gdf.iloc[0]
print(f"AOI: {aoi_row['NM_RGI']} (CD_RGI={aoi_row['CD_RGI']})")
print(f"Área: {float(aoi_row['AREA_KM2']):,.1f} km²")


## Dependência do mosaico Sentinel-2 (estágio 01)

Verifica (sem bloquear a execução) se o mosaico livre de nuvens exportado no estágio 01 está disponível no caminho canônico. As máscaras não dependem do arquivo, mas as etapas seguintes (03/04) sim.

In [ ]:
# Verifica se o mosaico exportado no estágio 01 está disponível no caminho canônico.
from src.data.gee_client import mosaic_file_name

mosaic_prefix = mosaic_file_name(
    config["aoi"]["region_code"],
    config["data"]["dates"]["start"],
    config["data"]["dates"]["end"],
)
mosaic_path = storage_paths["data_raw_sentinel2"] / f"{mosaic_prefix}.tif"

if io.path_exists(mosaic_path):
    print(f"Mosaico do estágio 01 disponível: {mosaic_path}")
else:
    print(f"Aviso: mosaico do estágio 01 ainda não encontrado em {mosaic_path}.")


## Fontes de ground truth habilitadas

Lista as fontes habilitadas em `src/config.yaml` e o ano de referência das máscaras (derivado das datas do mosaico), além de inicializar o registro de resultados das exportações.

In [ ]:
# Lista as fontes de ground truth habilitadas e o ano de referência das máscaras.
from pathlib import Path

from src.data.mask_utils import reference_year

year = reference_year()
enabled_sources = [
    name for name, cfg in config["ground_truth"]["sources"].items() if cfg["enabled"]
]
print(f"Ano de referência das máscaras: {year}")
print(f"Fontes habilitadas: {', '.join(enabled_sources) or 'nenhuma'}")
mask_results: dict[str, Path | None] = {}


## Máscara de café — MapBiomas

Constrói e exporta (idempotente) a máscara binária de café da MapBiomas — classe `46` (Café, 3.2.2.1) da Coleção 9 para o ano de referência — em `MyDrive/tcc/data/interim/mapbiomas/`. A coleção, a classe e a escala de processamento vêm de `src/config.yaml`.

In [ ]:
# Garante a máscara binária de café da MapBiomas (classe 46, Coleção 9).
from src.data.mask_utils import ensure_source_mask

mask_results["mapbiomas"] = ensure_source_mask("mapbiomas", aoi_geometry, storage_paths)


## Máscara de café — AlphaEarth

Constrói e exporta (idempotente) a máscara binária de café do modelo de probabilidade do Forest Data Partnership (coleção 2025a), derivado dos embeddings do AlphaEarth Foundations e rotulado `alphaearth-derived` no catálogo do GEE. O limiar de probabilidade vem de `src/config.yaml`; o resultado vai para `MyDrive/tcc/data/interim/alphaearth/`.

In [ ]:
# Garante a máscara binária de café do modelo de probabilidade da AlphaEarth (FDaP).
mask_results["alphaearth"] = ensure_source_mask("alphaearth", aoi_geometry, storage_paths)


## Pré-visualização das máscaras exportadas

Gera, para cada fonte habilitada com máscara disponível, uma miniatura binária (0/1) via Earth Engine e salva a figura em `MyDrive/tcc/artifacts/figures/` como registro visual da etapa. Verifica se a figura já existe antes de gerar novamente (idempotência).

In [ ]:
# Gera e salva a miniatura de cada máscara exportada apenas se ainda não existir (idempotente).
from src.data.mask_utils import mask_file_name, reference_year, save_mask_preview

for name, mask_path in mask_results.items():
    if mask_path is None:
        continue
    preview_path = (
        storage_paths["artifacts_figures"]
        / f"{mask_file_name(name, config['aoi']['region_code'], reference_year())}_preview.png"
    )
    save_mask_preview(name, aoi_geometry, preview_path)


## Resumo da etapa

Exibe o resumo da geração das máscaras: ano de referência, fontes habilitadas e o caminho de cada máscara exportada ou reutilizada.

In [ ]:
# Exibe o resumo da etapa de geração das máscaras de referência.
summary = {
    "Ano de referência": year,
    "Fontes habilitadas": ", ".join(enabled_sources) or "nenhuma",
}
for name, mask_path in mask_results.items():
    summary[f"Máscara {name}"] = str(mask_path) if mask_path else "desabilitada"

for key, value in summary.items():
    print(f"{key}: {value}")
print("Estágio 02 concluído.")
